In [1]:
%run data_ingestion.ipynb

c:\Users\AdedayoAdenrele\anaconda3\envs\Customer-churn\Lib\site-packages\fsspec\registry.py:301: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


In [2]:
df.duplicated()

0       False
1       False
2       False
3       False
4       False
        ...  
9995    False
9996    False
9997    False
9998    False
9999    False
Length: 10000, dtype: bool

In [3]:
df.isnull().sum()

customer_id           0
credit_score          0
country               0
gender                0
age                   0
customer_retention    0
balance               0
products_number       0
credit_card           0
active_member         0
estimated_salary      0
churn                 0
dtype: int64

In [4]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df["gender"] = encoder.fit_transform(df["gender"])

print(encoder.classes_)

['Female' 'Male']


**Female -> 0
Male   -> 1**


**Removing customer id**

In [5]:
df.drop(["customer_id"], axis=1, inplace=True)
df.drop(["country"], axis=1, inplace=True)

In [6]:
numerical_columns = [
    "credit_score",
    "age",
    "customer_retention",
    "balance",
    "products_number",
    "estimated_salary",
    "balance_salary_ratio"
]

In [7]:
categorical_columns = [
    "gender",
    "credit_card",
    "active_member",
    "active_card_user"
]

**Feature engineering**

In [8]:
df["balance_salary_ratio"] = (
    df["balance"] / (df["estimated_salary"] + 1)
)

**Splitting the data and scaling afterwards**

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
X=df.drop("churn", axis=1)
y=df["churn"]

X_train,X_test,y_train,y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
# keep unscaled copies for tree-based models (Random Forest, etc.) that
# don't need/benefit from feature scaling
X_train_raw = X_train.copy()
X_test_raw = X_test.copy()

scaler = StandardScaler()

X_train[numerical_columns] = scaler.fit_transform(X_train[numerical_columns])
X_test[numerical_columns] = scaler.transform(X_test[numerical_columns])

**Balancing the training set with SMOTE**

Churn is ~80/20 imbalanced. Resampling only `X_train`/`y_train` (never the test set) avoids leaking synthetic samples into evaluation.

In [10]:
from imblearn.over_sampling import SMOTENC

# gender / credit_card / active_member / active_card_user are categorical (0/1),
# so SMOTENC is used instead of plain SMOTE to avoid interpolating fractional
# values for those columns.
categorical_feature_indices = [
    X_train.columns.get_loc(col) for col in categorical_columns if col in X_train.columns
]

print("Class balance before SMOTE:")
print(y_train.value_counts(normalize=True))

# resample the raw (unscaled) features first, using the original y_train,
# for tree-based models like Random Forest
smote_raw = SMOTENC(categorical_features=categorical_feature_indices, random_state=42)
X_train_raw, y_train_raw = smote_raw.fit_resample(X_train_raw, y_train)

# resample the scaled features for models that need scaling (e.g. Logistic Regression)
smote = SMOTENC(categorical_features=categorical_feature_indices, random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

print("\nClass balance after SMOTE:")
print(y_train.value_counts(normalize=True))

Class balance before SMOTE:
churn
0    0.79625
1    0.20375
Name: proportion, dtype: float64

Class balance after SMOTE:
churn
1    0.5
0    0.5
Name: proportion, dtype: float64


In [11]:
import seaborn as sns

In [12]:
# sns.countplot(x="churn", data=df)